### Week 10 Homework Problem 1:

Use the `pokemon` dataset to create a Widget that displays the top 5 Pokemon (based on a user-selected characteristic) in a given `Type` selected by the user. The results should be displayed as a bar chart (with 5 bars, one for each Pokemon). The app should have two dropdowns: Type (Grass, Water, Fire, etc) and Stat (HP, Attack, Defense Special Attack, Special Defense, Speed). Also change the bars so the colors correspond to the current type.



In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import widgets
from ipywidgets import interact

In [2]:
data = pd.read_excel('pokemon.xlsx')

In [3]:
data.head()

,#,Name,Type,Total,HP,Attack,Defense,Special Attack,Special Defense,Speed
0,001,Bulbasaur,GRASS,318,45,49,49,65,65,45
1,001,Bulbasaur,POISON,318,45,49,49,65,65,45
2,002,Ivysaur,GRASS,405,60,62,63,80,80,60
3,002,Ivysaur,POISON,405,60,62,63,80,80,60
4,003,Venusaur,GRASS,525,80,82,83,100,100,80


In [4]:
#display top 5 pokemon based on a given stat
def top_pokemon(data, stat, n=5):
    return data.nlargest(n, stat)[['Name', stat]]

type_ = 'FIRE'
stat = 'Attack'
top_pokemon_data = data[data['Type'] == type_].nlargest(5, stat)[['Name', stat]]

In [5]:
top_pokemon_data

,Name,Attack
408,Mega Blaziken,160
882,Darmanitan- Standard Mode,140
883,Darmanitan- Standard Mode,140
12,Mega Charizard X,130
210,Flareon,130


There is an issue when having duplicates (name & stat, not index). The plot only shows four pokemon. I will have to create some sort of check to see if there are any duplicates and then add 6 pokemon instead of 5. This should fix the issue

In [8]:
def plot_top_pokemon(data, color_type, stat):
    # type changes the bar color
    type_colors = {
        'GRASS': 'green',
        'FIRE': 'red',
        'WATER': 'blue',
        'ELECTRIC': 'yellow',
        'ROCK': 'brown',
        'GROUND': 'gray',
        'POISON': 'purple',
        'BUG': 'lightgreen',
        'NORMAL': 'lightgray',
        'FAIRY': 'pink',
        'FIGHTING': 'orange',
        'PSYCHIC': 'magenta',
        'DRAGON': 'gold',
        'FLYING': 'lightblue',
        'STEEL': 'silver',
        'ICE': 'cyan',
        'GHOST': 'indigo',
        'DARK': 'black'
    }
    
    #filtering by type and sort by the highest stat
    top_pokemon_data = data[data['Type'] == color_type].nlargest(5, stat)[['Name', stat]]

    #check if there are duplicate names
    if top_pokemon_data['Name'].duplicated().any():
        #add another pokemon to make sure 5 are shown
        top_pokemon_data = data[data['Type'] == color_type].nlargest(6, stat)[['Name', stat]]

    #get color for type
    bar_color = type_colors.get(color_type)
    
    #plotting
    plt.figure(figsize=(10, 6))
    plt.bar(top_pokemon_data['Name'], top_pokemon_data[stat], color=bar_color)
    plt.xlabel('Pokemon')
    plt.ylabel(stat)
    plt.title(f'Top 5 Pokemon by Type')
    plt.show()

#dropdowns for type
type_dropdown = widgets.Dropdown(
    options=data['Type'].drop_duplicates().tolist()
)
#dropdown for stat
stat_dropdown = widgets.Dropdown(
    options=['HP', 'Attack', 'Defense', 'Special Attack', 'Special Defense', 'Speed']
)

# Create an interactive plot
interact(plot_top_pokemon, data=widgets.fixed(data), color_type=type_dropdown, stat=stat_dropdown)


interactive(children=(Dropdown(description='color_type', options=('GRASS', 'POISON', 'FIRE', 'FLYING', 'DRAGON…

<function __main__.plot_top_pokemon(data, color_type, stat)>

In [ ]:
#print the number of each type of pokemon
type_counts = data['Type'].value_counts()
print(type_counts)

### Week 10 Homework Problem 2:

Pivot tables are particularly useful for trajectory data (position over time for a moving object). The dataset various data of marine vessels over time including their latitude and longitude, course, speed and heading. Each ship is identified by a unique MMSI number. The data is currently formatted as stacked data. Use a pivot table to help you perform the following operations:
1. Plot the individual trajectories (x on the x axis, y on the y axis, ignore the time variable) for each ship on a single plot.
2. The plot above may look messy, as the ships could be very far apart. You are interested in the shape of individual trajectories, not their absolute positions. Create a new dataset of normalized trajectories, such that the average x and y position is equal to zero. Replot the normalized trajectories. Each trajectory should now be centered around the origin.
3. Calculate the speed and direction (as an angle between 0 and 360 degrees) for each ship at each time point. You should use a line drawn between the current point and the next point to calculate these values.
4. Create a widget that allows the user to select a ship and displays two histograms, one for the speed and one for the direction.


A few tips: use the `parse_dates` parameter for the `read_csv` method to interpret the `Date` column as dates instead of strings

Also think about the outputs you want when using `pivot`-- create two pivot tables, one for the `latitude` data and one for the `longitude` data.

In [9]:
import numpy as np
import pandas as pd
import seaborn as sns

In [10]:
df = pd.read_csv('shipTrajectories.csv')

#parse dates as dates
df['Time'] = pd.to_datetime(df['Time'])


In [11]:
df.head()

,Index,MMSI,Latitude,Longitude,Time
0,1,462803135,109.156070,19.789421,2019-06-02 05:44:00
1,2,462803135,109.151550,19.791151,2019-06-02 05:58:00
2,3,462803135,109.149640,19.791980,2019-06-02 06:00:00
3,4,462803135,109.146996,19.794230,2019-06-02 06:02:00
4,5,462803135,109.145730,19.795967,2019-06-02 06:04:00


In [12]:
df.dtypes

Index                 int64
MMSI                  int64
Latitude            float64
Longitude           float64
Time         datetime64[ns]
dtype: object

In [13]:
lat = df.melt(id_vars=['Index', 'MMSI'], var_name='Trajectory', value_name='Lat_Value')  # Use correct id_vars

In [14]:
lat.head()

,Index,MMSI,Trajectory,Lat_Value
0,1,462803135,Latitude,109.15607
1,2,462803135,Latitude,109.15155
2,3,462803135,Latitude,109.14964
3,4,462803135,Latitude,109.146996
4,5,462803135,Latitude,109.14573


In [ ]:
#plots strip plot
plot = sns.stripplot(x='MMSI', y='Lat_Value', data=lat)
plot.set_ylabel('Latitude')
plot.set_xlabel('MMSI')

In [ ]:
#create pivot table for longitude
lon = df.melt(id_vars=['Index', 'MMSI'], value_vars=['Longitude'], var_name='Trajectory', value_name='Long_Value')

In [ ]:
lon.head()

In [ ]:
#plots strip plot
plot = sns.stripplot(x='MMSI', y='Long_Value', data=lon)
plot.set_ylabel('Longitude')
plot.set_xlabel('MMSI')

### Problem 3 Reflection 

On every homework assignment. This should be at least 5 sentences and demonstrate engagement with the material that goes beyond "this was hard" or "this was cool." Things you may include are:

- What was most challenging and why
- How you approached a problem
- What you learned outside of class on your own
- How material complements what you are learning in other classes
- Feel free to take it any direction you want, but it needs to meet the above criteria to get full credit


Problem 1
- One of the challenges I faced was displaying the bar charts. I was able to pick out the 5 most powerful pokemon but the dataset had duplicate values. Because of that, when I displayed the pokemon, some would only have 4 bar plots. This is because the duplicate names and values would only be represent once even though they are two different rows/indexes. Because of that, They only showed 4 pokemon instead of 5. To get around that, I checked if there were any duplicated values and if there was, I would take in 6 pokemon instead of 5. That way, even if the barplot doesn't display the duplicate pokemon, there will be 5 pokemon no matter what. I learned that (I feel like I keep learning this lesson) that you have to work around datasets. There will be no perfect dataset. There will always be some cleaning/processing before making analysis.

Problem 2
- Idk, I'm working on it rn

# AI Acknowledgement Statement

Please indicate if and how you used ChatGPT or other LLMs on this assignment. Use the option that best fits your use case:

* [ ] I did not use ChatGPT/LLMs.
* [ ] I consulted ChatGPT/LLMs (to brainstorm, to debug, to get information, to give feedback), but I did not copy and paste or use text directly from ChatGPT/LLMs.
* [ ] I copied and pasted portions of code directly from ChatGPT/LLMs.
* [X] I consulted other outside sources other than ChatGPT/LLMs.

Should META pick ai development over mantiaining a larger workforce? Why or why not?
 - a

Do you think Ben & Jerry's should prioritize social mission over aligning buisness strategies of its parent company? Compromises, can both companies coexist?
 - a


If you were part of the pokemon community, how would you try to fix the problem? Can this plan be used world wide and not only at one location?
 - a

When do the dangers of flying outweigh the convenience?
 - a

How do you think forever 21s bankrupcy impacted the fast fashion industry? Did it signal a shift in consumer priorities?
- a


How can consumers help push companies toward more ethical practices?
- a


Do you think there should be stricter laws against using genetic information to sell/use data?
- a

As a manager, would you try to keep up with today's fashion trends and competitiors, or find a different angle?
- a


